# 08 — Explainable Conversational AI Interface (RAG + Control)

The Phase 2 explainability layer. An operator can ask, in natural language,
**why** the RL agent made its scaling decisions, and receive plain-English
answers grounded in the agent's actual decision log (Retrieval-Augmented
Generation). The same chat interface also accepts operator **commands** —
surge warnings that set the agent's hint slots (function calling).

Pipeline: decision log -> readable documents -> embeddings (all-MiniLM-L6-v2)
-> FAISS retrieval -> Mistral-7B (via Ollama) grounded generation -> Gradio UI.

**Requires Ollama running (`ollama serve`) with the `mistral` model pulled.**

Uses `env.py`, `agent.py`.

## 1. Generate a decision log from the trained agent

In [1]:
import json
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic

stats = json.load(open('trace_params.json'))['stats']

net = ActorCritic()
net.load_state_dict(torch.load('ppo_sla-focused.pth'))
net.eval()

# run the canonical agent through one week, capturing a rich decision log
env = CloudClusterEnv(stats, seed=123)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

decision_log = []
for t in range(STEPS_PER_WEEK):
    day, hour = env._current_day_hour()
    avg_cpu = env.history[-1][0]; avg_mem = env.history[-1][2]
    queue_before = len(env.queue); vms_before = env.active_vms
    with torch.no_grad():
        mean, _ = net.forward(obs_t.unsqueeze(0))
    obs, reward, done, tr, info = env.step(mean.squeeze(0).numpy())
    obs_t = torch.tensor(obs, dtype=torch.float32)
    vms_after = info['active_vms']; delta = vms_after - vms_before
    decision = (f"scaled UP by {delta}" if delta > 0
                else f"scaled DOWN by {abs(delta)}" if delta < 0 else "held steady")
    decision_log.append({
        'step': t, 'day': day, 'hour': hour,
        'cpu_load': round(float(avg_cpu),3), 'mem_load': round(float(avg_mem),3),
        'queue_before': queue_before, 'vms_before': vms_before,
        'vms_after': vms_after, 'decision': decision,
        'breaches': info['breaches'], 'cost': round(info['cost'],3),
        'utilisation': round(info['utilisation'],3)})
    if done: break

json.dump(decision_log, open('decision_log.json', 'w'), indent=2)
print(f"Captured {len(decision_log)} decisions. Saved decision_log.json")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Captured 672 decisions. Saved decision_log.json


## 2. Convert decisions to readable documents

In [2]:
def entry_to_text(e):
    return (f"On day {e['day']}, hour {e['hour']} (step {e['step']}): "
            f"CPU load was {e['cpu_load']:.0%}, memory load {e['mem_load']:.0%}, "
            f"with {e['queue_before']} jobs waiting in the queue. "
            f"The agent was running {e['vms_before']} VMs and {e['decision']}, "
            f"resulting in {e['vms_after']} VMs. "
            f"This step had {e['breaches']} SLA breaches, "
            f"cost {e['cost']:.2f}, and {e['utilisation']:.0%} utilisation.")

documents = [entry_to_text(e) for e in decision_log]
print(f"Created {len(documents)} documents. Example:")
print(" •", documents[1])

Created 672 documents. Example:
 • On day 0, hour 0 (step 1): CPU load was 100%, memory load 90%, with 336 jobs waiting in the queue. The agent was running 2 VMs and scaled UP by 5, resulting in 7 VMs. This step had 0 SLA breaches, cost 0.35, and 100% utilisation.


## 3. Embed documents and build the FAISS retrieval index

In [3]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(documents, show_progress_bar=True)

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(doc_embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} documents ({dim}-dim embeddings).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

FAISS index built with 672 documents (384-dim embeddings).


## 4. RAG generation (retrieve + Mistral grounded answer)

In [4]:
import ollama

def answer_question(question, k=5):
    q_vec = embed_model.encode([question]).astype('float32')
    _, idxs = index.search(q_vec, k=k)
    retrieved = [documents[i] for i in idxs[0]]
    context = "\n".join(f"- {d}" for d in retrieved)
    prompt = (f"You are an assistant that explains an AI cloud-scaling agent's decisions.\n"
              f"Answer the question using ONLY the decision records below. "
              f"Do not invent information.\n\n"
              f"Decision records:\n{context}\n\nQuestion: {question}\n\nAnswer:")
    resp = ollama.chat(model='mistral', messages=[{'role':'user','content':prompt}])
    return resp['message']['content'], retrieved

# capture a few example Q&A for the report
example_questions = [
    "Why did the agent scale up on day 3?",
    "Were there any SLA breaches this week?",
    "What did the agent do during quiet periods?",
]
qa_examples = []
for q in example_questions:
    ans, srcs = answer_question(q)
    qa_examples.append({'question': q, 'answer': ans, 'sources': srcs[:3]})
    print(f"Q: {q}\nA: {ans}\n{'-'*60}")

json.dump(qa_examples, open('rag_qa_examples.json', 'w'), indent=2)
print("Saved rag_qa_examples.json (for the report)")

Q: Why did the agent scale up on day 3?
A:  The AI cloud-scaling agent scaled up on day 3 at step 300 and again at step 334 due to high CPU load (100%) and a large number of jobs waiting in the queue (98 jobs). The agent added 2 VMs each time, increasing from 3 VMs to 5 VMs, to alleviate the excessive demand for resources.
------------------------------------------------------------
Q: Were there any SLA breaches this week?
A:  Yes, there were SLA breaches this week. Specifically, on day 6, hour 14 (step 634), the agent experienced 52 SLA breaches. Additionally, on the same day but in a different step (step 633), there were 42 SLA breaches. These two steps occurred within an hour of each other on the sixth day of the week.
------------------------------------------------------------
Q: What did the agent do during quiet periods?
A:  During quiet periods, as indicated by the CPU load and memory load being less than 100%, the agent held steady by running a smaller number of Virtual Machi

## 5. Function-calling control — routing + hint tool

In [5]:
import re

tools = [
    {'type':'function','function':{'name':'explain_decision',
        'description':'Explain why the agent made a decision, or answer any question '
                      'about its past behaviour, load, VMs, breaches, or cost.',
        'parameters':{'type':'object','properties':{
            'question':{'type':'string','description':"operator's question"}},
            'required':['question']}}},
    {'type':'function','function':{'name':'set_surge_hint',
        'description':'Warn the agent a traffic surge is expected so it can scale up '
                      'in advance. Use for upcoming spikes/surges/increased load.',
        'parameters':{'type':'object','properties':{
            'hour':{'type':'integer','description':'hour (0-23) of the surge'},
            'magnitude':{'type':'string','description':'small, medium, or large'}},
            'required':['hour','magnitude']}}},
]

SYSTEM_PROMPT = ("You are a control interface for an autonomous cloud-scaling agent. "
    "You MUST respond by calling exactly one tool. Never answer directly in plain text.\n"
    "- Questions about past behaviour/decisions/load/VMs/breaches/cost -> explain_decision.\n"
    "- Warnings about an upcoming surge/spike/increased traffic -> set_surge_hint.\n"
    "When extracting the hour, convert to 24-hour format (2pm = 14, 9am = 9).\n"
    "Always call a tool.")

def route_message(msg):
    r = ollama.chat(model='mistral',
        messages=[{'role':'system','content':SYSTEM_PROMPT},
                  {'role':'user','content':msg}], tools=tools)
    m = r['message']
    if m.get('tool_calls'):
        tc = m['tool_calls'][0]
        return tc['function']['name'], tc['function']['arguments']
    text = m.get('content','')
    for pat in [r'\{.*"name".*\}', r'\[\s*\{.*"name".*\}\s*\]']:
        mt = re.search(pat, text, re.DOTALL)
        if mt:
            try:
                p = json.loads(mt.group())
                if isinstance(p, list): p = p[0]
                if 'name' in p: return p['name'], p.get('arguments', {})
            except Exception: pass
    return None, text

operator_hints = []
def handle_message(msg):
    name, args = route_message(msg)
    if name == 'set_surge_hint':
        hour = int(np.clip(args.get('hour',0), 0, 23))
        mag = str(args.get('magnitude','medium'))
        operator_hints.append({'hour':hour,'magnitude':mag})
        return (f"✓ Surge hint registered: {mag} surge at hour {hour}. "
                f"It will be fed into the agent's hint inputs during the "
                f"lead-up window before hour {hour}. "
                f"(Active hints: {len(operator_hints)})")
    else:
        ans, _ = answer_question(args.get('question', msg) if isinstance(args, dict) else msg)
        return ans

# demonstrate both paths for the report
print("CONTROL TEST 1 (question):")
print(handle_message("Why did the agent scale up on day 3?"))
print("\nCONTROL TEST 2 (surge command):")
print(handle_message("Expect a large surge at 2pm"))

CONTROL TEST 1 (question):
 The AI cloud-scaling agent scaled up on day 3 at step 300 because the CPU load was 100%, memory load was 75%, and there were 98 jobs waiting in the queue while it was running 3 VMs. In this situation, with high CPU and memory loads and a large number of jobs waiting, the agent determined that scaling up by 2 VMs (from 3 to 5) would help reduce the load and prevent potential SLA breaches.

CONTROL TEST 2 (surge command):
✓ Surge hint registered: large surge at hour 14. It will be fed into the agent's hint inputs during the lead-up window before hour 14. (Active hints: 1)


## 5b. Executing the operator hint against the simulation

The registered hint is not just stored — it is applied to the agent's three
hint input slots during the lead-up window before the warned hour, using the
hint-aware model from notebook 06. Running the same seeded day with and
without the hint applied shows the agent provisioning ahead of the warned
hour, i.e. the chat command actually changes the agent's behaviour.

In [6]:
# Execute registered operator hints against the simulation.
# The hint-aware model (notebook 06) responds to the 3 hint slots; here the
# operator's chat command sets those slots in the live control loop.
from env import SURGE_LEAD_STEPS, STEPS_PER_HOUR

hint_net = ActorCritic()
hint_net.load_state_dict(torch.load('ppo_hint_aware.pth'))
hint_net.eval()

def inject_operator_hints(obs_t, next_step, hints):
    """Set the 3 hint slots when a registered surge hour is imminent.
    Magnitude is fixed at 1.0 — the training surge range saturates that slot,
    so the model responds to hint_active and time_to_event."""
    for h in hints:
        target_step = h['hour'] * STEPS_PER_HOUR      # first step of the warned hour
        steps_until = target_step - next_step
        if 0 < steps_until <= SURGE_LEAD_STEPS:
            obs_t[-3] = 1.0                            # hint_active
            obs_t[-2] = 1.0                            # hint_magnitude
            obs_t[-1] = 1.0 - steps_until / SURGE_LEAD_STEPS
    return obs_t

def run_day(apply_hints):
    e = CloudClusterEnv(stats, seed=777)
    obs, _ = e.reset()
    obs_t = torch.tensor(obs, dtype=torch.float32)
    vms = []
    for t in range(24 * STEPS_PER_HOUR):               # one simulated day
        if apply_hints:
            obs_t = inject_operator_hints(obs_t, e.step_count, operator_hints)
        with torch.no_grad():
            mean, _ = hint_net.forward(obs_t.unsqueeze(0))
        obs, r, done, tr, info = e.step(mean.squeeze(0).numpy())
        obs_t = torch.tensor(obs, dtype=torch.float32)
        vms.append(info['active_vms'])
    return vms

vms_hint = run_day(apply_hints=True)
vms_none = run_day(apply_hints=False)

hint_hour = operator_hints[-1]['hour'] if operator_hints else 14
target = hint_hour * STEPS_PER_HOUR
w0 = target - SURGE_LEAD_STEPS
print(f"Operator hint: surge warned for hour {hint_hour} "
      f"(hint slots active on steps {w0}-{target-1})\n")
print(f"{'step':>5} {'hour':>5} {'VMs with hint':>14} {'VMs without':>12}")
for t in range(max(w0 - 2, 0), min(target + 4, len(vms_hint))):
    mark = '  <- hint active' if w0 <= t < target else ''
    print(f"{t:>5} {t // STEPS_PER_HOUR:>5} {vms_hint[t]:>14} {vms_none[t]:>12}{mark}")

print(f"\nAvg VMs during lead window: with hint {np.mean(vms_hint[w0:target]):.1f}, "
      f"without {np.mean(vms_none[w0:target]):.1f}")

Operator hint: surge warned for hour 14 (hint slots active on steps 52-55)

 step  hour  VMs with hint  VMs without
   50    12              2            2
   51    12              2            2
   52    13              2            2  <- hint active
   53    13              5            4  <- hint active
   54    13              7            6  <- hint active
   55    13              8            7  <- hint active
   56    14              7            7
   57    14              8            8
   58    14              8            8
   59    14             10           10

Avg VMs during lead window: with hint 5.5, without 4.8


## 6. LLM-as-Judge evaluation — 25 curated questions

Formal evaluation of the conversational subsystem, as specified in the
dissertation outline: 25 test questions spanning **factual**, **causal**,
**predictive**, and **command-response** categories, scored with the
LLM-as-Judge methodology (Zheng et al., 2023 — reference [6]).

- Factual / causal / predictive questions go through the RAG pipeline and are
  scored 1–5 by a judge model on **groundedness** (does the answer stick to
  the retrieved records?), **correctness**, and **completeness**. The judge
  sees the question, the retrieved decision records, and the answer.
- Command-response items are scored **programmatically**: did the router call
  the correct tool with correctly extracted arguments (including 24-hour
  conversion)? Deterministic checks are more reliable than a judge for
  structured outputs.

Limitation (state this in the report): the judge is the same base model
(Mistral 7B) as the answerer, which risks self-preference bias. Set
`JUDGE_MODEL` to a different local model (e.g. `llama3`) if one is pulled.
Runtime: ~10–15 minutes of local inference.

In [7]:
# ---- 25 curated test questions --------------------------------------------
# RAG categories: judged 1-5. Commands: programmatic tool/argument checks.
TEST_QUESTIONS = [
    # factual (7)
    {'cat': 'factual', 'q': "How many VMs was the agent running at the start of the week?"},
    {'cat': 'factual', 'q': "What was the highest number of VMs the agent used during the week?"},
    {'cat': 'factual', 'q': "Were there any SLA breaches during the week?"},
    {'cat': 'factual', 'q': "What was the CPU load when the agent scaled up on day 3?"},
    {'cat': 'factual', 'q': "How many jobs were waiting in the queue at the busiest recorded moment?"},
    {'cat': 'factual', 'q': "What was the cost per step when the agent ran only 2 VMs?"},
    {'cat': 'factual', 'q': "On which day and hour did the largest single scale-up happen?"},
    # causal (7)
    {'cat': 'causal', 'q': "Why did the agent scale up on day 3?"},
    {'cat': 'causal', 'q': "Why did the agent scale down during the night hours?"},
    {'cat': 'causal', 'q': "Why did the agent hold steady instead of scaling during quiet periods?"},
    {'cat': 'causal', 'q': "What conditions led to SLA breaches this week?"},
    {'cat': 'causal', 'q': "Why did the agent keep utilisation high instead of adding more VMs?"},
    {'cat': 'causal', 'q': "What made the agent add VMs in the morning?"},
    {'cat': 'causal', 'q': "Why did the agent run only 2 VMs at some points in the week?"},
    # predictive (5)
    {'cat': 'predictive', 'q': "What would the agent likely do if CPU load reached 100% with a growing queue?"},
    {'cat': 'predictive', 'q': "How would the agent respond to a sudden traffic surge?"},
    {'cat': 'predictive', 'q': "Based on this week, what is the agent likely to do on Sunday night?"},
    {'cat': 'predictive', 'q': "If the queue doubled at midday, what response would you expect?"},
    {'cat': 'predictive', 'q': "Would the agent scale down if utilisation dropped very low?"},
    # command-response (6): expected tool + expected extracted hour
    {'cat': 'command', 'q': "Expect a large surge at 2pm",
     'tool': 'set_surge_hint', 'hour': 14},
    {'cat': 'command', 'q': "We're anticipating a big traffic spike at 9am",
     'tool': 'set_surge_hint', 'hour': 9},
    {'cat': 'command', 'q': "Warn the agent about a medium surge at 18:00",
     'tool': 'set_surge_hint', 'hour': 18},
    {'cat': 'command', 'q': "There will be increased load around 6 in the evening",
     'tool': 'set_surge_hint', 'hour': 18},
    {'cat': 'command', 'q': "Small spike expected at noon",
     'tool': 'set_surge_hint', 'hour': 12},
    {'cat': 'command', 'q': "Tell me why the agent reduced VMs overnight",
     'tool': 'explain_decision', 'hour': None},
]

# ---- judge -----------------------------------------------------------------
JUDGE_MODEL = 'mistral'   # swap for a different local model to reduce self-judging bias

JUDGE_PROMPT = """You are evaluating an AI assistant that answers questions about a
cloud-scaling agent, using ONLY retrieved decision records as its source.

Question: {question}

Retrieved decision records (the assistant's only allowed source):
{records}

Assistant's answer:
{answer}

Score the answer on three criteria, each an integer 1 (very poor) to 5 (excellent):
- groundedness: every claim is supported by the retrieved records; no invented details
- correctness: the answer is factually consistent with the records
- completeness: the answer addresses the question as fully as the records allow

Reply with ONLY a JSON object, no other text:
{{"groundedness": <1-5>, "correctness": <1-5>, "completeness": <1-5>, "justification": "<one sentence>"}}"""

def judge_answer(question, records, answer):
    prompt = JUDGE_PROMPT.format(
        question=question,
        records="\n".join(f"- {r}" for r in records),
        answer=answer)
    resp = ollama.chat(model=JUDGE_MODEL,
                       messages=[{'role': 'user', 'content': prompt}],
                       options={'temperature': 0.0})
    text = resp['message']['content']
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        s = json.loads(m.group())
        return {k: int(np.clip(int(s[k]), 1, 5))
                for k in ['groundedness', 'correctness', 'completeness']} | \
               {'justification': str(s.get('justification', ''))}
    except Exception:
        return None

# ---- run the evaluation ----------------------------------------------------
items = []
for i, t in enumerate(TEST_QUESTIONS):
    item = {'id': i + 1, 'category': t['cat'], 'question': t['q']}
    if t['cat'] == 'command':
        name, args = route_message(t['q'])
        args = args if isinstance(args, dict) else {}
        tool_ok = (name == t['tool'])
        if t['hour'] is None:
            args_ok = tool_ok           # explain_decision has no hour to check
        else:
            try:
                args_ok = int(args.get('hour', -1)) == t['hour']
            except (TypeError, ValueError):
                args_ok = False
        item |= {'expected_tool': t['tool'], 'called_tool': name,
                 'expected_hour': t['hour'], 'args': args,
                 'tool_correct': bool(tool_ok), 'args_correct': bool(args_ok)}
        print(f"[{i+1:>2}/25] command    tool {'OK ' if tool_ok else 'MISS'} "
              f"args {'OK ' if args_ok else 'MISS'}  {t['q']}")
    else:
        answer, retrieved = answer_question(t['q'])
        scores = judge_answer(t['q'], retrieved, answer)
        item |= {'answer': answer, 'sources': retrieved[:3], 'scores': scores}
        shown = (f"g{scores['groundedness']} c{scores['correctness']} "
                 f"m{scores['completeness']}") if scores else 'JUDGE PARSE FAIL'
        print(f"[{i+1:>2}/25] {t['cat']:<10} {shown}  {t['q']}")
    items.append(item)

# ---- summary ---------------------------------------------------------------
print("\n" + "=" * 66)
print("LLM-AS-JUDGE SUMMARY (25 questions)")
print("=" * 66)
print(f"{'category':<12}{'n':>3}{'groundedness':>14}{'correctness':>13}{'completeness':>14}")
summary = {}
for cat in ['factual', 'causal', 'predictive']:
    rows = [it['scores'] for it in items if it['category'] == cat and it.get('scores')]
    if rows:
        means = {k: float(np.mean([r[k] for r in rows]))
                 for k in ['groundedness', 'correctness', 'completeness']}
        summary[cat] = {'n': len(rows), **means}
        print(f"{cat:<12}{len(rows):>3}{means['groundedness']:>14.2f}"
              f"{means['correctness']:>13.2f}{means['completeness']:>14.2f}")
cmds = [it for it in items if it['category'] == 'command']
tool_acc = float(np.mean([it['tool_correct'] for it in cmds]))
args_acc = float(np.mean([it['args_correct'] for it in cmds]))
summary['command'] = {'n': len(cmds), 'tool_accuracy': tool_acc, 'args_accuracy': args_acc}
print(f"{'command':<12}{len(cmds):>3}   tool accuracy {tool_acc:.0%}, "
      f"argument accuracy {args_acc:.0%}")
unparsed = sum(1 for it in items if it['category'] != 'command' and not it.get('scores'))
if unparsed:
    print(f"\nNote: {unparsed} judge response(s) failed to parse (excluded from means).")

json.dump({'judge_model': JUDGE_MODEL, 'summary': summary, 'items': items},
          open('llm_judge_results.json', 'w'), indent=2)
print("\nSaved llm_judge_results.json")

[ 1/25] factual    g5 c5 m5  How many VMs was the agent running at the start of the week?
[ 2/25] factual    g5 c5 m5  What was the highest number of VMs the agent used during the week?
[ 3/25] factual    g5 c5 m5  Were there any SLA breaches during the week?
[ 4/25] factual    g5 c5 m5  What was the CPU load when the agent scaled up on day 3?
[ 5/25] factual    g5 c5 m5  How many jobs were waiting in the queue at the busiest recorded moment?
[ 6/25] factual    g5 c5 m5  What was the cost per step when the agent ran only 2 VMs?
[ 7/25] factual    g5 c5 m5  On which day and hour did the largest single scale-up happen?
[ 8/25] causal     g5 c5 m5  Why did the agent scale up on day 3?
[ 9/25] causal     g5 c5 m5  Why did the agent scale down during the night hours?
[10/25] causal     g5 c5 m5  Why did the agent hold steady instead of scaling during quiet periods?
[11/25] causal     g5 c5 m5  What conditions led to SLA breaches this week?
[12/25] causal     g5 c5 m4  Why did the agent keep

## 7. Unified Gradio chat interface (explain + control)

In [8]:
import gradio as gr

def chat_fn(message, history):
    return handle_message(message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Cloud Scaling Agent — Explain & Control Interface",
    description="Ask why the agent made decisions, or warn it about upcoming surges.",
    examples=["Why did the agent scale up on day 3?",
              "Were there any SLA breaches this week?",
              "Expect a large surge at 2pm",
              "When did the agent use the most VMs?"])
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
